# Train deepfake-detect on Kaggle

Trains the linear head on **FaceForensics++ (c23)** using the official train/val/test splits, then tests on **Celeb-DF v2**, a dataset the model never sees during training. That second number is the honest one.

**Before running**
1. Settings → Accelerator → **GPU T4 x2** (or P100). Turn **Internet on**.
2. Both datasets need an access request to their authors ([FaceForensics++](https://github.com/ondyari/FaceForensics), [Celeb-DF](https://github.com/yuezunli/celeb-deepfakeforensics)). Their licences forbid redistribution, so upload your copy as a **private** Kaggle dataset and attach it with *Add Input*.
3. Attach both datasets with *Add Input*. The notebook finds them under `/kaggle/input/` by itself.

Expected runtime: ~3 h, mostly video decoding and face detection. Embeddings are cached, so re-runs take minutes.

In [ ]:
REPO = "https://github.com/CodeRafay/deepfake-detect"
BRANCH = "main"
!git clone -q -b $BRANCH $REPO /kaggle/working/deepfake-detect
!pip install -q /kaggle/working/deepfake-detect pytest
!python -m pytest -q /kaggle/working/deepfake-detect/tests

In [ ]:
import json, urllib.request
from pathlib import Path

def find_input(name):
    """Kaggle mounts inputs at varying depths (e.g. /kaggle/input/datasets/<user>/<name>/), so search for a marker."""
    hits = sorted(Path("/kaggle/input").rglob(name))
    if not hits:
        raise FileNotFoundError(f"no '{name}' under /kaggle/input - is the dataset attached via Add Input?")
    return hits[0].parent

FFPP = find_input("original_sequences")              # folder holding original_sequences/ and manipulated_sequences/
CELEBDF = find_input("List_of_testing_videos.txt")   # folder holding Celeb-real/, Celeb-synthesis/, YouTube-real/
print("FF++:", FFPP)
print("Celeb-DF:", CELEBDF)
METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
DATA, CDF = Path("/kaggle/working/data"), Path("/kaggle/working/celebdf")

def link(src, dst):
    if src.exists() and not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.symlink_to(src)

# Official FF++ splits are by source-video pair, so no identity appears in two splits.
for split in ["train", "val", "test"]:
    url = f"https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/{split}.json"
    for a, b in json.load(urllib.request.urlopen(url)):
        for v in (a, b):
            link(FFPP / f"original_sequences/youtube/c23/videos/{v}.mp4", DATA / split / "real" / f"{v}.mp4")
        for m in METHODS:
            for x, y in ((a, b), (b, a)):
                link(FFPP / f"manipulated_sequences/{m}/c23/videos/{x}_{y}.mp4", DATA / split / "fake" / f"{m}_{x}_{y}.mp4")

# Celeb-DF official test list: "1 path" = real, "0 path" = fake.
for line in (CELEBDF / "List_of_testing_videos.txt").read_text().splitlines():
    if line.strip():
        lab, rel = line.split()
        link(CELEBDF / rel, CDF / ("real" if lab == "1" else "fake") / rel.replace("/", "_"))

for d in [DATA / s / c for s in ["train", "val", "test"] for c in ["real", "fake"]] + [CDF / "real", CDF / "fake"]:
    print(f"{d}: {len(list(d.glob('*')))} videos")

In [ ]:
!cd /kaggle/working && python -m deepfake_detect.train --data data --eval celebdf=celebdf --out head.safetensors

Paste the table above into the README's **Results** section. Report it as is, even if the Celeb-DF number is unflattering. Next, sanity-check a few test videos:

In [ ]:
from deepfake_detect import Detector
det = Detector("/kaggle/working/head.safetensors")
for p in sorted((CDF / "real").glob("*"))[:3] + sorted((CDF / "fake").glob("*"))[:3]:
    r = det.predict(p)
    print(p.parent.name, r["label"], r["score"])

## Publish models to Hugging Face (run on its own, after committing)
This cell is separate from training. Once you've put a new head in `models/<version>/` on GitHub and pushed it, run only this cell (no GPU needed). It copies the repo's `models/` folder to Hugging Face. Existing version folders are never deleted or replaced.

One-time setup: create a **Write** token at huggingface.co → Settings → Access Tokens, then add it here under *Add-ons → Secrets* with the label `HF_TOKEN`, attached to this notebook.

In [ ]:
REPO_URL = "https://github.com/CodeRafay/deepfake-detect"
BRANCH = "main"
HF_REPO = "CodeRafay/deepfake-detect"   # must match MODELS_REPO in deepfake_detect/model.py

!rm -rf /kaggle/working/publish && git clone -q --depth 1 -b $BRANCH $REPO_URL /kaggle/working/publish
!ls -R /kaggle/working/publish/models

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(folder_path="/kaggle/working/publish/models", repo_id=HF_REPO,
                  commit_message=f"sync models/ from GitHub {BRANCH}")
print(f"done: https://huggingface.co/{HF_REPO}")